In [ ]:
"""
Adaptive-Scale Satellite Image Embedding Experiment
Images sized according to census tract area
Naming: point{id}_tract{tract_id}_{lat}_{lon}.png  OR  point{id}_{lat}_{lon}.png
Models: ResNet50, CLIP, DINOv2-Small, DINOv2-Base, AlphaEarth
Regularization: L1, L2, ElasticNet for all image features
Metrics: AUC, Accuracy, Log-Likelihood reported for all models
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline
from scipy import stats
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# STEP 1: 数据加载
# ============================================================

print("="*70)
print("STEP 1: Loading Data")
print("="*70)

trip_df     = pd.read_csv('./chicago_trip_level_with_embeddings.csv')
idx_quarter = pd.read_csv('./satellite_images_quarter_mile/index_quarter_mile.csv')
idx_one     = pd.read_csv('./satellite_images_one_mile/index_one_mile.csv')


ADAPTIVE_DIR = Path('./satellite_images_adaptive1')  # 修改为你的实际路径

resnet_raw = np.load('./raw_resnet_embeddings.npy')
clip_raw   = np.load('./raw_clip_embeddings.npy')
emb_index  = pd.read_csv('./embedding_point_index.csv')

print(f"  Trips:            {len(trip_df):,}")
print(f"  Quarter-mile pts: {len(idx_quarter):,}")
print(f"  One-mile pts:     {len(idx_one):,}")
print(f"  0.5-mile pts:     {len(emb_index):,}")
print(f"  Adaptive dir:     {ADAPTIVE_DIR}")

# ============================================================
# STEP 2: 解析自适应图像目录，构建 index
# ============================================================

print("\n" + "="*70)
print("STEP 2: Parsing Adaptive Image Directory")
print("="*70)

def parse_adaptive_index(image_dir):
    """
    解析两种命名格式：
    格式A: point{id}_tract{tract_id}_{lat}_{lon}.png
    格式B: point{id}_{lat}_{lon}.png
    """
    image_dir = Path(image_dir)
    if not image_dir.exists():
        print(f"  [ERROR] Directory not found: {image_dir}")
        return pd.DataFrame()

    records = []
    png_files = list(image_dir.glob('*.png'))
    print(f"  Found {len(png_files)} PNG files")

    for fpath in png_files:
        name = fpath.stem   # 去掉 .png
        parts = name.split('_')

        try:
            # 找 point_id: 第一个 part 以 'point' 开头
            if not parts[0].startswith('point'):
                continue
            point_id = int(parts[0].replace('point', ''))

            # 判断是否有 tract
            if parts[1].startswith('tract'):
                tract_id = parts[1].replace('tract', '')
                lat = float(parts[2])
                lon = float(parts[3])
            else:
                tract_id = None
                lat = float(parts[1])
                lon = float(parts[2])

            records.append({
                'point_id':            point_id,
                'tract_id':            tract_id,
                'lat':                 lat,
                'lon':                 lon,
                'satellite_image_path': str(fpath),
            })

        except (ValueError, IndexError) as e:
            print(f"  [WARN] Cannot parse: {fpath.name} → {e}")
            continue

    df = pd.DataFrame(records)
    if len(df) > 0:
        df = df.sort_values('point_id').reset_index(drop=True)
        print(f"  Parsed: {len(df)} images")
        print(f"  With tract_id: {df['tract_id'].notna().sum()}")
        print(f"  Without tract_id: {df['tract_id'].isna().sum()}")
        print(f"  Lat range: [{df['lat'].min():.4f}, {df['lat'].max():.4f}]")
        print(f"  Sample:\n{df.head(3).to_string()}")
    return df

idx_adaptive = parse_adaptive_index(ADAPTIVE_DIR)

# ============================================================
# STEP 3: 依赖检查
# ============================================================

print("\n" + "="*70)
print("STEP 3: Checking Dependencies")
print("="*70)

def safe_import(pkg):
    try:
        __import__(pkg)
        print(f"  ✓ {pkg}")
        return True
    except ImportError:
        print(f"  ✗ {pkg}")
        return False

has_open_clip    = safe_import('open_clip')
has_timm         = safe_import('timm')
has_transformers = safe_import('transformers')

# ============================================================
# STEP 4: Dataset
# ============================================================

class SatelliteDataset(Dataset):
    """支持 satellite_image_path 列直接给路径"""

    def __init__(self, index_df, transform=None):
        self.index_df  = index_df.reset_index(drop=True)
        self.transform = transform

        self.valid_indices = []
        for i, row in self.index_df.iterrows():
            if Path(row['satellite_image_path']).exists():
                self.valid_indices.append(i)

        print(f"    Valid: {len(self.valid_indices)}/{len(self.index_df)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        row      = self.index_df.iloc[real_idx]
        image    = Image.open(row['satellite_image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {
            'image':    image,
            'lat':      float(row['lat']),
            'lon':      float(row['lon']),
            'point_id': int(row['point_id']),
        }


transform_standard = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

# ============================================================
# STEP 5: Embedding 提取器
# ============================================================

class EmbeddingExtractor:

    def __init__(self, model_name, device):
        self.model_name = model_name
        self.device     = device
        self.model      = None
        self.transform  = transform_standard
        self.embed_dim  = None
        self._is_clip   = False
        self._load()

    def _load(self):
        name = self.model_name
        print(f"\n  Loading [{name}] ...")

        if name == 'resnet50':
            base           = models.resnet50(weights='IMAGENET1K_V2')
            self.model     = nn.Sequential(*list(base.children())[:-1])
            self.embed_dim = 2048
            print("    ✓ ResNet-50")

        elif name == 'clip_vitb32':
            if has_open_clip:
                for pretrained in ['openai', 'laion2b_s34b_b79k']:
                    try:
                        import open_clip
                        m, _, prep = open_clip.create_model_and_transforms(
                            'ViT-B-32', pretrained=pretrained)
                        m = m.to(self.device).eval()
                        self.model     = m
                        self.transform = prep
                        self.embed_dim = 512
                        self._is_clip  = True
                        print(f"    ✓ CLIP ViT-B/32 ({pretrained})")
                        break
                    except Exception as e:
                        print(f"    · {pretrained} failed: {e}")

        elif name == 'dinov2_small':
            try:
                self.model     = torch.hub.load(
                    'facebookresearch/dinov2', 'dinov2_vits14', verbose=False)
                self.embed_dim = 384
                print("    ✓ DINOv2-Small (384-d)")
            except Exception as e:
                print(f"    ✗ {e}")

        elif name == 'dinov2_base':
            try:
                self.model     = torch.hub.load(
                    'facebookresearch/dinov2', 'dinov2_vitb14', verbose=False)
                self.embed_dim = 768
                print("    ✓ DINOv2-Base (768-d)")
            except Exception as e:
                print(f"    ✗ {e}")

        elif name == 'alphaearth':
            self._load_alphaearth()

        if self.model is not None and not self._is_clip:
            self.model = self.model.to(self.device).eval()
            print(f"    embed_dim = {self.embed_dim}")
        elif self.model is not None:
            print(f"    embed_dim = {self.embed_dim}")

    def _load_alphaearth(self):
        if has_transformers:
            try:
                from transformers import AutoModel
                m = AutoModel.from_pretrained(
                    "ibm-nasa-geospatial/Prithvi-100M",
                    trust_remote_code=True)
                self.model     = m.to(self.device).eval()
                self.embed_dim = 768
                print("    ✓ AlphaEarth → Prithvi-100M")
                return
            except Exception as e:
                print(f"    · Prithvi failed: {e}")

        if has_timm:
            try:
                import timm
                m = timm.create_model(
                    'swin_base_patch4_window7_224',
                    pretrained=True, num_classes=0)
                self.model     = m.to(self.device).eval()
                self.embed_dim = 1024
                print("    ✓ AlphaEarth → Swin-B (timm)")
                return
            except Exception as e:
                print(f"    · Swin-B failed: {e}")

        if has_open_clip:
            try:
                import open_clip
                m, _, prep = open_clip.create_model_and_transforms(
                    'ViT-L-14', pretrained='laion2b_s32b_b82k')
                m = m.to(self.device).eval()
                self.model     = m
                self.transform = prep
                self.embed_dim = 768
                self._is_clip  = True
                print("    ✓ AlphaEarth → OpenCLIP ViT-L/14")
                return
            except Exception as e:
                print(f"    · OpenCLIP failed: {e}")

        print("    ✗ All AlphaEarth alternatives failed")

    @torch.no_grad()
    def extract(self, index_df, batch_size=32):
        if self.model is None:
            return None

        dataset = SatelliteDataset(index_df, transform=self.transform)
        if len(dataset) == 0:
            return None

        loader = DataLoader(dataset, batch_size=batch_size,
                            shuffle=False, num_workers=4,
                            pin_memory=(self.device == 'cuda'))

        all_emb, all_lat, all_lon, all_pid = [], [], [], []

        for batch in loader:
            imgs = batch['image'].to(self.device).float()

            if self._is_clip:
                emb = self.model.encode_image(imgs)
                emb = emb / emb.norm(dim=-1, keepdim=True)
                emb = emb.float()
            elif 'dinov2' in self.model_name:
                emb = self.model(imgs)
            elif self.model_name == 'alphaearth' and has_transformers:
                try:
                    imgs6 = imgs.repeat(1, 2, 1, 1)[:, :6]
                    out   = self.model(pixel_values=imgs6)
                    emb   = out.last_hidden_state[:, 0, :]
                except Exception:
                    try:
                        out = self.model(pixel_values=imgs)
                        emb = out.last_hidden_state[:, 0, :]
                    except Exception:
                        emb = self.model(imgs)
                        if emb.dim() > 2:
                            emb = emb.flatten(1)
            else:
                emb = self.model(imgs)
                if emb.dim() > 2:
                    emb = emb.flatten(1)

            all_emb.append(emb.cpu().float().numpy())
            all_lat.extend(batch['lat'].tolist())
            all_lon.extend(batch['lon'].tolist())
            all_pid.extend(batch['point_id'].tolist())

        return {
            'embeddings': np.vstack(all_emb),
            'lats':       np.array(all_lat),
            'lons':       np.array(all_lon),
            'point_ids':  np.array(all_pid),
        }

# ============================================================
# STEP 6: 提取所有尺度的 Embeddings
# ============================================================

print("\n" + "="*70)
print("STEP 6: Extracting Embeddings")
print("="*70)

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"  Device: {device}")

CACHE_DIR = Path('./embedding_cache')
CACHE_DIR.mkdir(exist_ok=True)

MODELS_TO_TEST = [
    'resnet50',
    'clip_vitb32',
    'dinov2_small',
    'dinov2_base',
    'alphaearth',
]

# 新图像尺度（需要提取的）
NEW_SCALES = {
    'quarter_mile': idx_quarter,
    'one_mile':     idx_one,
}

# 如果有自适应图像，加进去
if len(idx_adaptive) > 0:
    NEW_SCALES['adaptive'] = idx_adaptive

new_embeddings = {}

for model_name in MODELS_TO_TEST:
    new_embeddings[model_name] = {}
    extractor = EmbeddingExtractor(model_name, device)

    if extractor.model is None:
        print(f"  → Skipping {model_name}")
        del extractor
        continue

    for scale, index_df in NEW_SCALES.items():
        if len(index_df) == 0:
            continue

        cache_emb  = CACHE_DIR / f'{model_name}_{scale}_emb.npy'
        cache_meta = CACHE_DIR / f'{model_name}_{scale}_meta.csv'

        if cache_emb.exists() and cache_meta.exists():
            print(f"\n  [cache] {model_name} @ {scale}")
            meta   = pd.read_csv(cache_meta)
            result = {
                'embeddings': np.load(cache_emb),
                'lats':       meta['lat'].values,
                'lons':       meta['lon'].values,
                'point_ids':  meta['point_id'].values,
            }
            print(f"    shape: {result['embeddings'].shape}")
        else:
            print(f"\n  Extracting {model_name} @ {scale} ...")
            result = extractor.extract(index_df, batch_size=32)
            if result is None:
                continue
            np.save(cache_emb, result['embeddings'])
            pd.DataFrame({
                'lat':      result['lats'],
                'lon':      result['lons'],
                'point_id': result['point_ids'],
            }).to_csv(cache_meta, index=False)
            print(f"    shape: {result['embeddings'].shape} → cached")

        new_embeddings[model_name][scale] = result

    del extractor
    if device == 'cuda':
        torch.cuda.empty_cache()

print("\n[✓] Embeddings done")

# ============================================================
# STEP 7: 整合所有尺度
# ============================================================

print("\n" + "="*70)
print("STEP 7: Integrating All Scales")
print("="*70)

if 'point_id' not in emb_index.columns:
    emb_index['point_id'] = np.arange(len(emb_index))

half_base = {
    'lats':      emb_index['lat'].values,
    'lons':      emb_index['lon'].values,
    'point_ids': emb_index['point_id'].values,
}

# 构建统一数据结构
all_scale_emb = {
    '0.5_mile': {
        'resnet50':    {**half_base, 'embeddings': resnet_raw},
        'clip_vitb32': {**half_base, 'embeddings': clip_raw},
    },
    'quarter_mile': {},
    'one_mile':     {},
}

if len(idx_adaptive) > 0:
    all_scale_emb['adaptive'] = {}

for model_name in new_embeddings:
    for scale in new_embeddings[model_name]:
        all_scale_emb[scale][model_name] = new_embeddings[model_name][scale]

print("\n  Available combinations:")
for scale, md in all_scale_emb.items():
    for mn, r in md.items():
        print(f"    {scale:<15} × {mn:<15}: {r['embeddings'].shape}")

# ============================================================
# STEP 8: Tabular 特征
# ============================================================

print("\n" + "="*70)
print("STEP 8: Tabular Features")
print("="*70)

person_hh = [c for c in [
    'age','Female','License','Employed','Student',
    'White','Black','Asian','Hispanic','Bachelor_Above',
    'hhveh','Zero_Vehicle_HH','hhsize',
    'Low_Income','High_Income','Home_Owner',
] if c in trip_df.columns]

trip_f  = [c for c in ['od_dist_mi'] if c in trip_df.columns]
be_vars = [c for c in trip_df.columns
           if (c.startswith('O_') or c.startswith('D_'))
           and '_pc' not in c
           and trip_df[c].notna().mean() > 0.5]

tabular_feats = person_hh + trip_f + be_vars
y_col         = 'is_transit'

df = trip_df.dropna(subset=tabular_feats + [y_col]).copy()
for col in person_hh + trip_f:
    df = df[df[col] >= 0]

print(f"  person_hh: {len(person_hh)}  BE: {len(be_vars)}  "
      f"total: {len(tabular_feats)}")
print(f"  Trips: {len(df):,}  transit: {df[y_col].mean()*100:.1f}%")

# ============================================================
# STEP 9: KD-Tree 坐标匹配
# ============================================================

print("\n" + "="*70)
print("STEP 9: KD-Tree Coordinate Matching")
print("="*70)

def match_trips_kd(df_trips, emb_result, max_dist_deg=0.005):
    coords = np.column_stack([emb_result['lats'], emb_result['lons']])
    tree   = cKDTree(coords)
    oq     = df_trips[['orig_lat', 'orig_lon']].values
    dq     = df_trips[['dest_lat', 'dest_lon']].values
    od, oi = tree.query(oq, k=1)
    dd, di = tree.query(dq, k=1)
    valid  = (od < max_dist_deg) & (dd < max_dist_deg)
    return oi, di, valid

match_info = {}
print(f"\n  {'Scale':<15} {'Model':<15} {'Matched':>10} {'Rate':>8}")
print(f"  {'-'*55}")

for scale, md in all_scale_emb.items():
    for mn, emb_result in md.items():
        key       = f"{scale}__{mn}"
        oi, di, v = match_trips_kd(df, emb_result)
        match_info[key] = {'orig_idx': oi, 'dest_idx': di, 'valid': v}
        print(f"  {scale:<15} {mn:<15} {v.sum():>10,} {v.mean()*100:>7.1f}%")

all_valid = np.ones(len(df), dtype=bool)
for info in match_info.values():
    all_valid &= info['valid']

print(f"\n  Intersection: {all_valid.sum():,} trips")

df_valid  = df[all_valid].reset_index(drop=True)
y_all     = df_valid[y_col].values
X_tab_raw = df_valid[tabular_feats].values

for key in match_info:
    match_info[key]['o_sub'] = match_info[key]['orig_idx'][all_valid]
    match_info[key]['d_sub'] = match_info[key]['dest_idx'][all_valid]

print(f"  Final: {len(df_valid):,}  transit: {y_all.mean()*100:.1f}%")

# ============================================================
# STEP 10: Train/Test Split
# ============================================================

train_idx, test_idx = train_test_split(
    np.arange(len(df_valid)), test_size=0.2,
    stratify=y_all, random_state=42)

y_train, y_test = y_all[train_idx], y_all[test_idx]

scaler_tab  = StandardScaler()
X_tab_train = scaler_tab.fit_transform(X_tab_raw[train_idx])
X_tab_test  = scaler_tab.transform(X_tab_raw[test_idx])

m1_idx = [tabular_feats.index(c)
          for c in (person_hh + trip_f) if c in tabular_feats]
m2_idx = list(range(len(tabular_feats)))

print(f"\n  Train: {len(train_idx):,}  Test: {len(test_idx):,}")
print(f"  M1 (no BE): {len(m1_idx)} feats")
print(f"  M2 (all):   {len(m2_idx)} feats")

# ============================================================
# STEP 11: 辅助函数
# ============================================================

PCA_DIMS = [5, 10, 20, 50]

# 正则化配置：所有图像特征都用 L1 / L2 / ElasticNet
REG_CONFIGS = [
    ('L2',         'l2',         'lbfgs',    {}),
    ('L1',         'l1',         'liblinear', {}),
    ('ElasticNet', 'elasticnet', 'saga',      {'l1_ratio': 0.5}),
]

C_VALUES = [0.01, 0.1, 0.5, 1.0]   # 用交叉验证选最佳 C


def get_od_pca(scale, model_name, n_pca):
    """OD PCA，fit on train only"""
    key = f"{scale}__{model_name}"
    emb = all_scale_emb[scale][model_name]['embeddings']
    ov  = match_info[key]['o_sub']
    dv  = match_info[key]['d_sub']
    O, D = emb[ov], emb[dv]

    OD_tr = np.vstack([O[train_idx], D[train_idx]])
    sc    = StandardScaler()
    OD_sc = sc.fit_transform(OD_tr)
    nc    = min(n_pca, OD_sc.shape[0], OD_sc.shape[1])
    pca   = PCA(n_components=nc, random_state=42)
    pca.fit(OD_sc)
    var   = pca.explained_variance_ratio_.sum() * 100

    Otr = pca.transform(sc.transform(O[train_idx]))
    Ote = pca.transform(sc.transform(O[test_idx]))
    Dtr = pca.transform(sc.transform(D[train_idx]))
    Dte = pca.transform(sc.transform(D[test_idx]))

    return (np.column_stack([Otr, Dtr]),
            np.column_stack([Ote, Dte]),
            var, nc)


def compute_metrics(y_tr, y_te, y_prob):
    """统一计算 AUC / Accuracy / Log-Likelihood / ρ²"""
    auc  = roc_auc_score(y_te, y_prob)
    acc  = accuracy_score(y_te, (y_prob >= 0.5).astype(int))
    eps  = 1e-15
    p    = np.clip(y_prob, eps, 1 - eps)
    ll   = float(np.sum(y_te * np.log(p) + (1 - y_te) * np.log(1 - p)))
    p0   = y_tr.mean()
    ll0  = float(np.sum(y_te * np.log(p0) + (1 - y_te) * np.log(1 - p0)))
    rho2 = 1 - ll / ll0
    return {'auc': auc, 'acc': acc, 'll': ll, 'rho2': rho2}


def evaluate_with_reg(X_tr, y_tr, X_te, y_te,
                      reg_name, penalty, solver, extra_kwargs,
                      C=0.5):
    """带正则化的 Logistic Regression 评估"""
    kwargs = dict(C=C, penalty=penalty, solver=solver,
                  max_iter=3000, random_state=42)
    kwargs.update(extra_kwargs)

    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('lr', LogisticRegression(**kwargs)),
    ])
    try:
        pipe.fit(X_tr, y_tr)
        y_prob = pipe.predict_proba(X_te)[:, 1]
        return compute_metrics(y_tr, y_te, y_prob)
    except Exception as e:
        return None


def best_reg_result(X_tr, y_tr, X_te, y_te):
    """
    对 L1 / L2 / ElasticNet × C_VALUES 网格搜索，
    返回每种正则化的最佳结果（按 AUC 选 C）
    """
    best = {}
    for reg_name, penalty, solver, extra in REG_CONFIGS:
        best_auc = -1
        best_r   = None
        best_C   = None
        for C in C_VALUES:
            r = evaluate_with_reg(
                X_tr, y_tr, X_te, y_te,
                reg_name, penalty, solver, extra, C)
            if r is not None and r['auc'] > best_auc:
                best_auc = r['auc']
                best_r   = r
                best_C   = C
        if best_r is not None:
            best[reg_name] = {**best_r, 'best_C': best_C}
    return best


def print_reg_block(config_name, reg_results,
                    ref_ll=None, ref_auc=None, indent="  "):
    """打印一个 config 的 L1/L2/EN 三行结果"""
    print(f"{indent}{'Config':<45} {'Reg':<12} {'C':>6} "
          f"{'AUC':>8} {'Acc':>7} {'LL':>12} {'ρ²':>8}"
          + (f" {'ΔAUC':>8} {'ΔLL':>8}" if ref_ll is not None else ""))
    print(f"{indent}{'-'*105}")
    for reg_name, r in reg_results.items():
        d_auc = f"{r['auc']-ref_auc:>+8.4f}" if ref_auc else ""
        d_ll  = f"{r['ll'] -ref_ll:>+8.1f}"  if ref_ll  else ""
        print(f"{indent}{config_name:<45} {reg_name:<12} {r['best_C']:>6.3f} "
              f"{r['auc']:>8.4f} {r['acc']:>7.4f} {r['ll']:>12.1f} "
              f"{r['rho2']:>8.4f}{d_auc}{d_ll}")


def do_lrt(ll_r, ll_f, df_diff):
    s   = max(2 * (ll_f - ll_r), 0)
    d   = abs(df_diff)
    pv  = 1 - stats.chi2.cdf(s, d) if d > 0 else 1.0
    sig = ('***' if pv < 0.001 else '**' if pv < 0.01
           else '*' if pv < 0.05 else 'ns')
    return s, pv, sig

# ============================================================
# EXPERIMENT A: Tabular Baselines（含正则化）
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT A: Tabular Baselines with Regularization")
print("="*70)

tab_r = {}
for tab_name, feat_idx in [
    ('M1: Person+HH+Trip (no BE)', m1_idx),
    ('M2: Full Tabular (+BE)',      m2_idx),
]:
    print(f"\n  [{tab_name}]")
    Xtr = X_tab_train[:, feat_idx]
    Xte = X_tab_test[:,  feat_idx]
    reg_res = best_reg_result(Xtr, y_train, Xte, y_test)
    print_reg_block(tab_name, reg_res)
    tab_r[tab_name] = reg_res

# 取 L2 结果作为参考（和之前保持一致）
m1_auc = tab_r['M1: Person+HH+Trip (no BE)']['L2']['auc']
m2_auc = tab_r['M2: Full Tabular (+BE)']['L2']['auc']
m1_ll  = tab_r['M1: Person+HH+Trip (no BE)']['L2']['ll']
m2_ll  = tab_r['M2: Full Tabular (+BE)']['L2']['ll']
m1_acc = tab_r['M1: Person+HH+Trip (no BE)']['L2']['acc']
m2_acc = tab_r['M2: Full Tabular (+BE)']['L2']['acc']

print(f"\n  Reference (L2): M1 AUC={m1_auc:.4f} Acc={m1_acc:.4f} LL={m1_ll:.1f}")
print(f"  Reference (L2): M2 AUC={m2_auc:.4f} Acc={m2_acc:.4f} LL={m2_ll:.1f}")

# ============================================================
# EXPERIMENT B: Image-Only（Scale × Model × PCA × Reg）
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT B: Image-Only  (Scale × Model × PCA × Regularization)")
print("="*70)

img_only   = {}   # key=(scale, model, nc*2, reg_name)
img_only_b = {}   # key=(scale, model, nc*2) → best across regs

for scale, md in all_scale_emb.items():
    print(f"\n  === Scale: {scale} ===")
    for mn in md:
        for np_ in PCA_DIMS:
            try:
                Xtr, Xte, var, nc = get_od_pca(scale, mn, np_)
            except Exception as e:
                continue

            reg_res = best_reg_result(Xtr, y_train, Xte, y_test)
            label   = f"ImgOnly_{mn}@{scale}_PCA{nc*2}"
            print_reg_block(label, reg_res)

            for reg_name, r in reg_res.items():
                img_only[(scale, mn, nc*2, reg_name)] = r

            # 记录最佳（L2 for comparability）
            if 'L2' in reg_res:
                img_only_b[(scale, mn, nc*2)] = reg_res['L2']

# ============================================================
# EXPERIMENT C: Scale Comparison (PCA=20, best reg)
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT C: Scale Comparison (PCA=20, L2 reg)")
print("  Which spatial scale is most informative?")
print("="*70)
print(f"\n  {'Model':<15} {'Scale':<15} {'AUC':>8} {'Acc':>7} "
      f"{'LL':>12} {'ρ²':>8}")
print(f"  {'-'*70}")

for mn in ['resnet50','clip_vitb32','dinov2_small',
           'dinov2_base','alphaearth']:
    for scale in ['0.5_mile','quarter_mile','one_mile','adaptive']:
        k = (scale, mn, 40)   # PCA=20 → nc*2=40
        if k in img_only_b:
            r = img_only_b[k]
            print(f"  {mn:<15} {scale:<15} {r['auc']:>8.4f} "
                  f"{r['acc']:>7.4f} {r['ll']:>12.1f} {r['rho2']:>8.4f}")
    print()


# ============================================================
# EXPERIMENT D: Multi-Scale Fusion（Image-Only）
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT D: Multi-Scale Fusion (Image-Only, PCA=20, L2)")
print("="*70)
print(f"\n  {'Config':<50} {'AUC':>8} {'Acc':>7} {'LL':>12} "
      f"{'ΔvsBest_AUC':>12} {'ΔvsBest_LL':>11}")
print(f"  {'-'*105}")

available_scales = list(all_scale_emb.keys())

for mn in ['resnet50','clip_vitb32','dinov2_small',
           'dinov2_base','alphaearth']:

    feats = {}
    for scale in available_scales:
        if mn not in all_scale_emb.get(scale, {}):
            continue
        try:
            Xtr, Xte, _, _ = get_od_pca(scale, mn, 20)
            feats[scale]   = (Xtr, Xte)
        except Exception:
            continue

    if len(feats) < 2:
        continue

    # 单尺度（L2）
    single = {}
    for scale, (Xtr, Xte) in feats.items():
        r = best_reg_result(Xtr, y_train, Xte, y_test).get('L2', {})
        single[scale] = r
        print(f"  {mn+' '+scale:<50} {r['auc']:>8.4f} "
              f"{r['acc']:>7.4f} {r['ll']:>12.1f} "
              f"{'(single)':>12} {'(single)':>11}")

    best_auc = max(v['auc'] for v in single.values())
    best_ll  = max(v['ll']  for v in single.values())

    # 两两融合
    sl = list(feats.keys())
    for i in range(len(sl)):
        for j in range(i+1, len(sl)):
            s1, s2 = sl[i], sl[j]
            Xtr2   = np.column_stack([feats[s1][0], feats[s2][0]])
            Xte2   = np.column_stack([feats[s1][1], feats[s2][1]])
            r2_all = best_reg_result(Xtr2, y_train, Xte2, y_test)
            r2     = r2_all.get('L2', {})
            b_auc  = max(single[s1]['auc'], single[s2]['auc'])
            b_ll   = max(single[s1]['ll'],  single[s2]['ll'])
            lbl    = f"{mn} {s1[:8]}+{s2[:8]}"

            delta_auc = r2['auc'] - b_auc
            delta_ll  = r2['ll']  - b_ll
            d_auc_str = f"D{delta_auc:+.4f}"   # D 代替 Δ 避免编码问题
            d_ll_str  = f"D{delta_ll:+.1f}"

            print(f"  {lbl:<50} {r2['auc']:>8.4f} "
                  f"{r2['acc']:>7.4f} {r2['ll']:>12.1f} "
                  f"{d_auc_str:>12} {d_ll_str:>11}")

    # 全部融合
    if len(feats) >= 3:
        Xtr_a  = np.column_stack([v[0] for v in feats.values()])
        Xte_a  = np.column_stack([v[1] for v in feats.values()])
        ra_all = best_reg_result(Xtr_a, y_train, Xte_a, y_test)
        ra     = ra_all.get('L2', {})

        delta_auc_a = ra['auc'] - best_auc
        delta_ll_a  = ra['ll']  - best_ll
        da_auc_str  = f"D{delta_auc_a:+.4f}"
        da_ll_str   = f"D{delta_ll_a:+.1f}"

        print(f"  {mn+' ALL-FUSED':<50} {ra['auc']:>8.4f} "
              f"{ra['acc']:>7.4f} {ra['ll']:>12.1f} "
              f"{da_auc_str:>12} {da_ll_str:>11}")
    print()
# ============================================================
# EXPERIMENT E: Full Model — Tab + Image (含正则化)
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT E: Full Model — Tabular + Image with Regularization")
print("  M1+Image vs M2(+BE)  →  Can imagery substitute BE?")
print("="*70)

all_full = []

for scale, md in all_scale_emb.items():
    print(f"\n  === Scale: {scale} ===")
    for mn in md:
        for np_ in [5, 10, 20]:
            try:
                Xtr_img, Xte_img, var, nc = get_od_pca(scale, mn, np_)
            except Exception:
                continue

            # M1 + Image
            Xtra    = np.column_stack([X_tab_train[:, m1_idx], Xtr_img])
            Xtea    = np.column_stack([X_tab_test[:,  m1_idx], Xte_img])
            ra_all  = best_reg_result(Xtra, y_train, Xtea, y_test)

            # M2 + Image
            Xtrb    = np.column_stack([X_tab_train, Xtr_img])
            Xteb    = np.column_stack([X_tab_test,  Xte_img])
            rb_all  = best_reg_result(Xtrb, y_train, Xteb, y_test)

            # 打印 M1+Image
            label_a = f"M1+{mn}@{scale}_PCA{nc*2}"
            print(f"\n  [{label_a}]  (ΔvsM1 / ΔvsM2)")
            print(f"  {'Reg':<12} {'C':>6} {'AUC':>8} {'Acc':>7} "
                  f"{'LL':>12} {'ρ²':>8} "
                  f"{'ΔAUC_M1':>9} {'ΔAUC_M2':>9} "
                  f"{'ΔLL_M1':>9} {'ΔLL_M2':>9}")
            print(f"  {'-'*100}")

            for reg_name, ra in ra_all.items():
                dm1a = ra['auc'] - m1_auc
                dm2a = ra['auc'] - m2_auc
                dm1l = ra['ll']  - m1_ll
                dm2l = ra['ll']  - m2_ll
                print(f"  {reg_name:<12} {ra['best_C']:>6.3f} "
                      f"{ra['auc']:>8.4f} {ra['acc']:>7.4f} "
                      f"{ra['ll']:>12.1f} {ra['rho2']:>8.4f} "
                      f"{dm1a:>+9.4f} {dm2a:>+9.4f} "
                      f"{dm1l:>+9.1f} {dm2l:>+9.1f}")
                all_full.append({
                    'config':       label_a,
                    'tab_type':     'M1_noBE',
                    'scale':        scale,
                    'model':        mn,
                    'n_pca':        nc*2,
                    'reg':          reg_name,
                    'best_C':       ra['best_C'],
                    'auc':          ra['auc'],
                    'acc':          ra['acc'],
                    'll':           ra['ll'],
                    'rho2':         ra['rho2'],
                    'delta_m1_auc': dm1a,
                    'delta_m2_auc': dm2a,
                    'delta_m1_ll':  dm1l,
                    'delta_m2_ll':  dm2l,
                })

            # 打印 M2+Image
            label_b = f"M2+{mn}@{scale}_PCA{nc*2}"
            print(f"\n  [{label_b}]  (ΔvsM1 / ΔvsM2)")
            print(f"  {'Reg':<12} {'C':>6} {'AUC':>8} {'Acc':>7} "
                  f"{'LL':>12} {'ρ²':>8} "
                  f"{'ΔAUC_M1':>9} {'ΔAUC_M2':>9} "
                  f"{'ΔLL_M1':>9} {'ΔLL_M2':>9}")
            print(f"  {'-'*100}")

            for reg_name, rb in rb_all.items():
                dm1a = rb['auc'] - m1_auc
                dm2a = rb['auc'] - m2_auc
                dm1l = rb['ll']  - m1_ll
                dm2l = rb['ll']  - m2_ll
                print(f"  {reg_name:<12} {rb['best_C']:>6.3f} "
                      f"{rb['auc']:>8.4f} {rb['acc']:>7.4f} "
                      f"{rb['ll']:>12.1f} {rb['rho2']:>8.4f} "
                      f"{dm1a:>+9.4f} {dm2a:>+9.4f} "
                      f"{dm1l:>+9.1f} {dm2l:>+9.1f}")
                all_full.append({
                    'config':       label_b,
                    'tab_type':     'M2_withBE',
                    'scale':        scale,
                    'model':        mn,
                    'n_pca':        nc*2,
                    'reg':          reg_name,
                    'best_C':       rb['best_C'],
                    'auc':          rb['auc'],
                    'acc':          rb['acc'],
                    'll':           rb['ll'],
                    'rho2':         rb['rho2'],
                    'delta_m1_auc': dm1a,
                    'delta_m2_auc': dm2a,
                    'delta_m1_ll':  dm1l,
                    'delta_m2_ll':  dm2l,
                })

# ============================================================
# EXPERIMENT F: Likelihood Ratio Tests
# ============================================================

print("\n" + "="*70)
print("EXPERIMENT F: Likelihood Ratio Tests")
print("="*70)

best_img_key = max(img_only_b, key=lambda k: img_only_b[k]['auc'])
bs, bm, bnf  = best_img_key
best_img_ll  = img_only_b[best_img_key]['ll']
print(f"\n  Best img-only: {bm}@{bs} PCA={bnf}  "
      f"AUC={img_only_b[best_img_key]['auc']:.4f}")

eps     = 1e-15
p0      = y_train.mean()
ll_null = float(np.sum(
    y_test*np.log(p0+eps) + (1-y_test)*np.log(1-p0+eps)))

Xtr_bi, Xte_bi, _, _ = get_od_pca(bs, bm, 20)

r_m1i = best_reg_result(
    np.column_stack([X_tab_train[:, m1_idx], Xtr_bi]), y_train,
    np.column_stack([X_tab_test[:,  m1_idx], Xte_bi]), y_test
).get('L2', {})

r_m2i = best_reg_result(
    np.column_stack([X_tab_train, Xtr_bi]), y_train,
    np.column_stack([X_tab_test,  Xte_bi]), y_test
).get('L2', {})

lrt_tests = [
    ('Null  → Image-only', ll_null,        best_img_ll,    bnf + 1),
    ('Null  → M1',         ll_null,        m1_ll,          len(m1_idx) + 1),
    ('Null  → M2',         ll_null,        m2_ll,          len(m2_idx) + 1),
    ('M1    → M1+Image',   m1_ll,          r_m1i.get('ll', m1_ll), bnf),
    ('M2    → M2+Image',   m2_ll,          r_m2i.get('ll', m2_ll), bnf),
]

print(f"\n  {'Test':<28} {'ΔLL':>8} {'df':>5} "
      f"{'χ²':>10} {'p':>12} {'sig':>5}")
print(f"  {'-'*73}")
for nm, ll_r, ll_f, ddf in lrt_tests:
    s, pv, sig = do_lrt(ll_r, ll_f, ddf)
    print(f"  {nm:<28} {ll_f-ll_r:>8.1f} {abs(ddf):>5} "
          f"{s:>10.1f} {pv:>12.6f} {sig:>5}")

# ============================================================
# SUMMARY & SAVE
# ============================================================

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

rdf = pd.DataFrame(all_full)
rdf.to_csv('./scale_model_results_adaptive.csv', index=False)

# Scale winner (L2, PCA=20)
print("\n  [Scale winner per model — image-only L2, PCA=20]")
print(f"  {'Model':<15} {'0.5mi':>10} {'quarter':>10} "
      f"{'one-mile':>10} {'adaptive':>10}  {'Winner'}")
print(f"  {'-'*70}")
for mn in ['resnet50','clip_vitb32','dinov2_small',
           'dinov2_base','alphaearth']:
    row = {}
    for s in ['0.5_mile','quarter_mile','one_mile','adaptive']:
        k = (s, mn, 40)
        if k in img_only_b:
            row[s] = img_only_b[k]['auc']
    if not row:
        continue
    best = max(row, key=row.get)
    vs   = [f"{row.get(s, float('nan')):>10.4f}"
            for s in ['0.5_mile','quarter_mile','one_mile','adaptive']]
    print(f"  {mn:<15} {'  '.join(vs)}  {best}")

# BE substitution — Top-5 M1+Image (L2)
print("\n  [Top-5 M1+Image L2: Can imagery substitute BE?]")
if len(rdf) > 0:
    top = (rdf[(rdf['tab_type'] == 'M1_noBE') & (rdf['reg'] == 'L2')]
           .nlargest(5, 'auc'))
    if len(top) > 0:
        print(f"\n  {'Config':<50} {'AUC':>8} {'Acc':>7} {'LL':>12} "
              f"{'ΔAUC_M1':>9} {'ΔAUC_M2':>9} {'ΔLL_M1':>9} {'ΔLL_M2':>9}")
        print(f"  {'-'*115}")
        for _, row in top.iterrows():
            print(f"  {row['config']:<50} {row['auc']:>8.4f} "
                  f"{row['acc']:>7.4f} {row['ll']:>12.1f} "
                  f"{row['delta_m1_auc']:>+9.4f} {row['delta_m2_auc']:>+9.4f} "
                  f"{row['delta_m1_ll']:>+9.1f} {row['delta_m2_ll']:>+9.1f}")
        print(f"\n  M1 (no BE): AUC={m1_auc:.4f} Acc={m1_acc:.4f} LL={m1_ll:.1f}")
        print(f"  M2 (+BE):   AUC={m2_auc:.4f} Acc={m2_acc:.4f} LL={m2_ll:.1f}")

# Model ranking
print("\n  [Overall model ranking — image-only best AUC (L2)]")
model_best = {}
for (scale, mn, nf), r in img_only_b.items():
    if mn not in model_best or r['auc'] > model_best[mn]['auc']:
        model_best[mn] = {**r, 'scale': scale, 'n_feat': nf}
for rank, (mn, r) in enumerate(
        sorted(model_best.items(),
               key=lambda x: x[1]['auc'], reverse=True), 1):
    print(f"  {rank}. {mn:<15} AUC={r['auc']:.4f} "
          f"Acc={r['acc']:.4f} LL={r['ll']:.1f}  "
          f"best_scale={r['scale']}")

print(f"\n  Saved → ./scale_model_results_adaptive.csv")
print("\n[✓] All experiments complete!")

STEP 1: Loading Data
  Trips:            20,815
  Quarter-mile pts: 1,148
  One-mile pts:     1,148
  0.5-mile pts:     1,147
  Adaptive dir:     satellite_images_adaptive1

STEP 2: Parsing Adaptive Image Directory
  Found 1148 PNG files
  Parsed: 1148 images
  With tract_id: 387
  Without tract_id: 761
  Lat range: [41.5093, 42.4368]
  Sample:
   point_id tract_id        lat        lon                                        satellite_image_path
0         1     None  41.891022 -87.612931  satellite_images_adaptive1/point1_41.891022_-87.612931.png
1         2     None  41.928424 -87.684906  satellite_images_adaptive1/point2_41.928424_-87.684906.png
2         3     None  41.895035 -87.619717  satellite_images_adaptive1/point3_41.895035_-87.619717.png

STEP 3: Checking Dependencies
  ✓ open_clip
  ✓ timm
  ✓ transformers

STEP 6: Extracting Embeddings
  Device: cuda

  Loading [resnet50] ...
    ✓ ResNet-50
    embed_dim = 2048

  [cache] resnet50 @ quarter_mile
    shape: (1148, 2048)

 